# CV02 — Khải: Huấn luyện mô hình cơ sở XGBoost Regressor

**Thành viên phụ trách:** Khải  
**Đầu vào:** `data/processed/train.csv`, `test.csv` (từ CV01)  
**Đầu ra:**  
- `outputs/models/xgboost_baseline.json`  
- `outputs/figures/cv02_learning_curve.png`  
- `outputs/tables/cv02_train_history.csv`

---

## 1. Cấu hình môi trường & đường dẫn

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from xgboost import XGBRegressor

NOTEBOOK_DIR  = Path().resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
TRAIN_DATA    = PROJECT_ROOT / 'data' / 'processed' / 'train.csv'
TEST_DATA     = PROJECT_ROOT / 'data' / 'processed' / 'test.csv'
MODELS_DIR    = PROJECT_ROOT / 'outputs' / 'models'
FIGURES_DIR   = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_DIR    = PROJECT_ROOT / 'outputs' / 'tables'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'AEP_MW'
XGB_PARAMS = {
    'n_estimators': 1000,
    'learning_rate': 0.01,
    'max_depth': 5,
    'early_stopping_rounds': 50,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'random_state': 42,
    'n_jobs': -1,
}
print('✅ Cấu hình hoàn tất')

## 2. Nạp dữ liệu từ CV01 & tách X – y

In [ ]:
def load_df(path):
    df = pd.read_csv(path)
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df.set_index('Datetime', inplace=True)
    return df

train_df = load_df(TRAIN_DATA)
test_df  = load_df(TEST_DATA)

features   = [c for c in train_df.columns if c != TARGET_COL]
X_train, y_train = train_df[features], train_df[TARGET_COL]
X_test,  y_test  = test_df[features],  test_df[TARGET_COL]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Đặc trưng ({len(features)}): {features}')

## 3. Huấn luyện XGBoost Baseline (Bước 5 đề bài)

In [ ]:
model = XGBRegressor(**XGB_PARAMS)

t0 = time.perf_counter()
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)
elapsed = time.perf_counter() - t0

best_iter  = int(model.best_iteration)
best_score = float(model.best_score)
print(f'\n⏱  Thời gian huấn luyện : {elapsed:.1f}s')
print(f'🏆  Best iteration      : {best_iter + 1}')
print(f'📉  Test RMSE tốt nhất  : {best_score:,.4f} MW')

## 4. Biểu đồ Learning Curve (RMSE theo vòng lặp)

In [ ]:
evals = model.evals_result()
train_rmse = evals['validation_0']['rmse']
test_rmse  = evals['validation_1']['rmse']

history = pd.DataFrame({
    'iteration': range(len(train_rmse)),
    'train_rmse': train_rmse,
    'test_rmse':  test_rmse,
})
history.to_csv(TABLES_DIR / 'cv02_train_history.csv', index=False)
print('✅ Đã lưu cv02_train_history.csv')

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(history['iteration'] + 1, history['train_rmse'], label='Train RMSE', color='#2a78d6', lw=2)
ax.plot(history['iteration'] + 1, history['test_rmse'],  label='Test RMSE',  color='#eb6834', lw=2, ls='--')
ax.axvline(best_iter + 1, color='grey', ls=':', lw=1.2, label=f'Best: {best_iter + 1}')
ax.scatter([best_iter + 1], [best_score], color='#eb6834', s=60, zorder=3)
ax.annotate(f'RMSE={best_score:,.0f}', xy=(best_iter+1, best_score), xytext=(10,8),
            textcoords='offset points', fontsize=9)
ax.set(title='Đường cong mất mát XGBoost Baseline', xlabel='Vòng lặp boosting', ylabel='RMSE (MW)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()

fig_path = FIGURES_DIR / 'cv02_learning_curve.png'
fig.savefig(fig_path, dpi=150)
plt.show()
print(f'✅ Đã lưu: {fig_path}')

## 5. Lưu checkpoint mô hình → bàn giao cho CV03

In [ ]:
model_path = MODELS_DIR / 'xgboost_baseline.json'
model.save_model(str(model_path))
size_kb = model_path.stat().st_size / 1024
print(f'✅ Checkpoint đã lưu: {model_path}  ({size_kb:.1f} KB)')
print('📦 Bàn giao cho CV03: xgboost_baseline.json sẵn sàng.')

## ✅ CV02 Hoàn tất

| Sản phẩm | Đường dẫn |
|:---|:---|
| Model checkpoint | `outputs/models/xgboost_baseline.json` |
| Learning curve | `outputs/figures/cv02_learning_curve.png` |
| Lịch sử RMSE | `outputs/tables/cv02_train_history.csv` |